In [1]:
from ucimlrepo import fetch_ucirepo 
  
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from pandas import DataFrame
import numpy as np
from torch.nn import functional as F
from torchinfo import summary
import mlflow
import pandas as pd
from torchtyping import TensorType, patch_typeguard
from typeguard import typechecked
from typing import Tuple

# fetch dataset 
iris = fetch_ucirepo(id=53) 
  
# data (as pandas dataframes) 
X = iris.data.features 
y = iris.data.targets 
  
# metadata 
print(iris.metadata) 
  
# variable information 
print(iris.variables) 


{'uci_id': 53, 'name': 'Iris', 'repository_url': 'https://archive.ics.uci.edu/dataset/53/iris', 'data_url': 'https://archive.ics.uci.edu/static/public/53/data.csv', 'abstract': 'A small classic dataset from Fisher, 1936. One of the earliest known datasets used for evaluating classification methods.\n', 'area': 'Biology', 'tasks': ['Classification'], 'characteristics': ['Tabular'], 'num_instances': 150, 'num_features': 4, 'feature_types': ['Real'], 'demographics': [], 'target_col': ['class'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 1936, 'last_updated': 'Tue Sep 12 2023', 'dataset_doi': '10.24432/C56C76', 'creators': ['R. A. Fisher'], 'intro_paper': {'ID': 191, 'type': 'NATIVE', 'title': 'The Iris data set: In search of the source of virginica', 'authors': 'A. Unwin, K. Kleinman', 'venue': 'Significance, 2021', 'year': 2021, 'journal': 'Significance, 2021', 'DOI': '1740-9713.01589', 'URL': 'https://www.semanticscholar.org

In [13]:
X.sample(5)

,sepal length,sepal width,petal length,petal width
73,6.1,2.8,4.7,1.2
61,5.9,3.0,4.2,1.5
35,5.0,3.2,1.2,0.2
81,5.5,2.4,3.7,1.0
30,4.8,3.1,1.6,0.2


In [2]:
mlflow.set_tracking_uri(uri="http://192.168.100.203:5000/")
mlflow.set_experiment("[P] Classification with Iris Dataset")

dataset = mlflow.data.from_pandas(
   pd.concat([X, y], axis=1), name=iris.metadata["name"], targets=iris.metadata.target_col[0]
)


In [3]:
def preprocess(X: DataFrame, y: DataFrame) -> tuple:
    assert isinstance(X, DataFrame)
    assert isinstance(y, DataFrame)
    assert X.shape[1] >= 1
    assert y.shape[1] == 1

    y_vector: np.ndarray = y.to_numpy().ravel()
    label_encoder = LabelEncoder()
    label_encoder.fit(y_vector)

    mapping = {
        "columns": {idx: col for idx, col in enumerate(X.columns)},
        "labels": {idx: label for idx, label in enumerate(label_encoder.classes_)},
    }

    X_processed: np.ndarray = X.to_numpy()
    y_processed: np.ndarray = label_encoder.transform(y_vector)

    return (
        torch.tensor(X_processed, dtype=torch.float32),
        torch.tensor(y_processed, dtype=torch.long),
        mapping,
    )


In [27]:
class Classifier(nn.Module):
    def __init__(self, input_size=4, hidden_size=16, output_size=3):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.fc1(x)
        return self.out(x)


X_p, y_p, mapping = preprocess(X, y)
X_train, X_test, y_train, y_test = train_test_split(
    X_p, y_p, test_size=0.20, random_state=1
)

criterion = nn.CrossEntropyLoss()
model = Classifier()
optimizer = optim.Adam(model.parameters(), lr=0.01)
NUM_EPOCHS = 100

params = {
    "input_size": X_train.shape[1],
    "hidden_size": 16,
    "output_size": len(mapping["labels"]),
    "learning_rate": 0.01,
    "epochs": NUM_EPOCHS,
    "criterion": criterion,
}
TOTAL_SAMPLES = X.shape[0]
TEST_SIZE_RATIO = 0.20
TRAIN_BATCH_SIZE = 120
TEST_BATCH_SIZE = round(TOTAL_SAMPLES * TEST_SIZE_RATIO)
NUM_CLASSES = 3
NUM_FEATURES = 4

patch_typeguard()  # Patch typeguard to work with torchtyping


@typechecked
def train(
    model: nn.Module,
    X_train: TensorType[TRAIN_BATCH_SIZE, NUM_FEATURES, torch.float32],
    y_train: TensorType[TRAIN_BATCH_SIZE, torch.long],
    criterion: nn.Module,
    optimizer: optim.Optimizer,
) -> Tuple[nn.Module, float, float]:
    model.train()
    optimizer.zero_grad()
    outputs: TensorType[TRAIN_BATCH_SIZE, NUM_CLASSES, torch.float32] = model(X_train)
    loss: TensorType[torch.float32] = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()

    train_predicted = torch.argmax(outputs, dim=1).to(torch.float32)
    train_accuracy: float = (train_predicted == y_train).float().mean().item()

    return model, loss.item(), train_accuracy


@typechecked
def evaluate(
    model: nn.Module,
    X_test: TensorType[TEST_BATCH_SIZE, NUM_FEATURES, torch.float32],
    y_test: TensorType[TEST_BATCH_SIZE, torch.long],
    criterion: nn.Module,
) -> Tuple[float, float]:
    model.eval()
    with torch.no_grad():
        outputs: TensorType[TEST_BATCH_SIZE, NUM_CLASSES, torch.float32] = model(X_test)
        test_loss: TensorType[torch.float32] = criterion(outputs, y_test).item()
        test_predicted = torch.argmax(outputs, dim=1).to(torch.float32)
        test_accuracy: float = (test_predicted == y_test).float().mean().item()

    return test_loss, test_accuracy


with mlflow.start_run(log_system_metrics=True) as run:
    mlflow.log_params(params)
    mlflow.log_input(dataset, context="training", tags={"source": "UCI ML Repository"})

    mlflow.set_tag("purpose", "configuration")
    mlflow.set_tag("framework", "pytorch")
    mlflow.set_tag("task", "classification")

    for epoch in range(NUM_EPOCHS):
        model, loss, train_accuracy = train(
            model, X_train, y_train, criterion, optimizer
        )
        mlflow.log_metrics(
            {
                "train_loss": loss,
                "train_accuracy": train_accuracy,
            },
            step=epoch,
        )

        test_loss, test_accuracy = evaluate(model, X_test, y_test, criterion)
        mlflow.log_metrics(
            {
                "test_loss": test_loss,
                "test_accuracy": test_accuracy,
            },
            step=epoch,
        )

    model.eval()
    with open("model_summary.txt", "w") as f:
        f.write(
            str(
                summary(
                    model,
                    device="cpu",
                    input_data=X_train[0],
                    col_names=[
                        "input_size",
                        "output_size",
                        "num_params",
                        "params_percent",
                        "mult_adds",
                        "kernel_size"
                    ],
                    verbose=2
                )
            )
        )

    with open("model_architecture.txt", "w") as f:
        f.write(str(model))
    mlflow.pytorch.log_model(
        model,
        name="Iris Classifier",
        input_example=X_train.numpy()[0],
        extra_files=["model_summary.txt", "model_architecture.txt"],
    )

2025/07/01 12:44:46 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2025/07/01 12:44:46 WARNING mlflow.system_metrics.metrics.gpu_monitor: Encountered error Not Supported when trying to collect GPU power usage metrics.


Layer (type:depth-idx)                   Input Shape               Output Shape              Param #                   Param %                   Mult-Adds                 Kernel Shape
Classifier                               [4]                       [3]                       --                             --                   --                        --
├─Linear: 1-1                            [4]                       [16]                      80                         61.07%                   1,280                     --
│    └─weight                                                                                ├─64                                                                          [4, 16]
│    └─bias                                                                                  └─16                                                                          [16]
├─Linear: 1-2                            [16]                      [3]                       51                  

2025/07/01 12:44:54 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/07/01 12:44:54 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2025/07/01 12:44:54 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


🏃 View run amazing-donkey-372 at: http://192.168.100.203:5000/#/experiments/3/runs/ea8c2e472cb54b42b57d03f2f625ef54
🧪 View experiment at: http://192.168.100.203:5000/#/experiments/3
